# Amazon — EmbeddingANN Optimised (Optuna)

**Goal**: Find best EmbeddingANN hyperparameters using Optuna TPE sampler.

**Search space:**
| Parameter | Range |
|-----------|-------|
| `user_emb_dim` | 4, 8, 16, 32 |
| `prod_emb_dim` | 4, 8, 16, 32 |
| `n_layers` | 1, 2, 3 |
| `hidden_dim` | 32 – 256 |
| `emb_dropout` | 0.1 – 0.6 |
| `mlp_dropout` | 0.1 – 0.5 |
| `lr` | 1e-4 – 1e-2 (log) |
| `batch_size` | 256, 512, 1024 |
| `weight_decay` | 1e-6 – 1e-3 (log) |

**Reads from**: `data/` + `embeddings/` (encoders only — re-learns weights)
**Saves to**  : `optimization_ANN/`


## Roadmap
```
STEP 1  — Imports & paths
STEP 2  — Load data & encoders
STEP 3  — Filter active IDs (same MIN_REVIEWS=5 as EmbeddingANN)
STEP 4  — Scale numerical features
STEP 5  — Dataset & DataLoader factory
STEP 6  — Model definition (flexible n_layers)
STEP 7  — Optuna HPO
STEP 8  — Retrain final model on best params
STEP 9  — Evaluate
STEP 10 — Save best embeddings & results
```

---
## Step 1 — Imports & Paths

In [ ]:
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import optuna
from optuna.pruners  import MedianPruner
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED   = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED); np.random.seed(SEED)
print(f"Device: {DEVICE}")
print(f"Optuna: {optuna.__version__}")


In [ ]:
DATA_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\\data"
EMB_DIR  = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\\embeddings"
OUT_DIR  = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\optimization_ANN"
os.makedirs(OUT_DIR, exist_ok=True)


---
## Step 2 — Load Data & Encoders

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train_features.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val_features.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test_features.csv'))
print(f"Train : {df_train.shape}  Val : {df_val.shape}  Test : {df_test.shape}")


In [ ]:
FEATURE_COLS = [
    'user_avg_rating', 'user_rating_count',
    'product_avg_rating', 'product_rating_count', 'product_rating_std',
    'days_since_review'
]
TARGET = 'ratings'


In [ ]:
# Load encoders only — we re-learn the embedding weights during HPO
with open(os.path.join(EMB_DIR, 'user_encoder.json'))    as f: user2idx    = json.load(f)
with open(os.path.join(EMB_DIR, 'product_encoder.json')) as f: product2idx = json.load(f)
print(f"User vocab    : {len(user2idx):,}")
print(f"Product vocab : {len(product2idx):,}")
n_users    = len(user2idx)
n_products = len(product2idx)


---
## Step 3 — Encode IDs

In [ ]:
def encode_ids(df, user2idx, product2idx):
    u = df['userId'].map(lambda x: user2idx.get(x, 0)).values.astype(np.int64)
    p = df['productId'].map(lambda x: product2idx.get(x, 0)).values.astype(np.int64)
    return u, p

train_user_idx, train_prod_idx = encode_ids(df_train, user2idx, product2idx)
val_user_idx,   val_prod_idx   = encode_ids(df_val,   user2idx, product2idx)
test_user_idx,  test_prod_idx  = encode_ids(df_test,  user2idx, product2idx)
print("IDs encoded.")


---
## Step 4 — Scale Numerical Features

In [ ]:
scaler      = StandardScaler()
X_train_num = scaler.fit_transform(df_train[FEATURE_COLS]).astype(np.float32)
X_val_num   = scaler.transform(df_val[FEATURE_COLS]).astype(np.float32)
X_test_num  = scaler.transform(df_test[FEATURE_COLS]).astype(np.float32)

y_train = df_train[TARGET].values.astype(np.float32)
y_val   = df_val[TARGET].values.astype(np.float32)
y_test  = df_test[TARGET].values.astype(np.float32)


---
## Step 5 — Dataset & DataLoader Factory

In [ ]:
class AmazonDataset(Dataset):
    def __init__(self, num_arr, user_idx, prod_idx, labels):
        self.num      = torch.tensor(num_arr,  dtype=torch.float32)
        self.user_idx = torch.tensor(user_idx, dtype=torch.long)
        self.prod_idx = torch.tensor(prod_idx, dtype=torch.long)
        self.labels   = torch.tensor(labels,   dtype=torch.float32)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return self.num[idx], self.user_idx[idx], self.prod_idx[idx], self.labels[idx]

def make_loaders(batch_size):
    tr = DataLoader(AmazonDataset(X_train_num, train_user_idx, train_prod_idx, y_train),
                    batch_size=batch_size, shuffle=True)
    v  = DataLoader(AmazonDataset(X_val_num,   val_user_idx,   val_prod_idx,   y_val),
                    batch_size=batch_size, shuffle=False)
    te = DataLoader(AmazonDataset(X_test_num,  test_user_idx,  test_prod_idx,  y_test),
                    batch_size=batch_size, shuffle=False)
    return tr, v, te


---
## Step 6 — Model Definition (Flexible Architecture)

Unlike the previous fixed 2-layer MLP, Optuna now also tunes
the number of hidden layers (1–3). Each layer has the same width
for simplicity.


In [ ]:
class FlexEmbeddingANN(nn.Module):
    def __init__(self, n_users, n_products, n_num,
                 user_emb_dim, prod_emb_dim,
                 n_layers, hidden_dim,
                 emb_dropout, mlp_dropout, weight_decay=1e-5):
        super().__init__()
        self.user_emb    = nn.Embedding(n_users,    user_emb_dim, padding_idx=0)
        self.prod_emb    = nn.Embedding(n_products, prod_emb_dim, padding_idx=0)
        self.emb_dropout = nn.Dropout(emb_dropout)
        self.n_users     = n_users
        self.n_products  = n_products

        # Build MLP with variable number of layers
        mlp_input = user_emb_dim + prod_emb_dim + n_num
        layers = []
        in_dim = mlp_input
        for _ in range(n_layers):
            layers += [nn.Linear(in_dim, hidden_dim), nn.ReLU(), nn.Dropout(mlp_dropout)]
            in_dim  = hidden_dim
        layers += [nn.Linear(in_dim, 1)]
        self.mlp = nn.Sequential(*layers)

    def forward(self, num, user_idx, prod_idx):
        user_idx = user_idx.clamp(0, self.n_users    - 1)
        prod_idx = prod_idx.clamp(0, self.n_products - 1)
        u = self.emb_dropout(self.user_emb(user_idx))
        p = self.emb_dropout(self.prod_emb(prod_idx))
        return self.mlp(torch.cat([u, p, num], dim=1)).squeeze(1)


---
## Step 7 — Optuna HPO

Each trial:
1. Suggests hyperparameters
2. Trains for N_EPOCHS_TRIAL epochs with early stopping
3. Returns best val RMSE

MedianPruner stops bad trials early based on intermediate val RMSE reports.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    model.train() if optimizer else model.eval()
    preds_all, labels_all = [], []
    ctx = torch.enable_grad() if optimizer else torch.no_grad()
    with ctx:
        for num_b, u_b, p_b, y_b in loader:
            num_b,u_b,p_b,y_b = num_b.to(DEVICE),u_b.to(DEVICE),p_b.to(DEVICE),y_b.to(DEVICE)
            preds = model(num_b, u_b, p_b)
            loss  = nn.MSELoss()(preds, y_b)
            if optimizer:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            preds_all.extend(preds.detach().cpu().numpy())
            labels_all.extend(y_b.cpu().numpy())
    p = np.array(preds_all); l = np.array(labels_all)
    return round(float(np.sqrt(mean_squared_error(l, p))), 4)


In [ ]:
N_EPOCHS_TRIAL = 10
PATIENCE_TRIAL = 3

def objective(trial):
    # Suggest hyperparameters
    user_emb_dim = trial.suggest_categorical('user_emb_dim', [4, 8, 16, 32])
    prod_emb_dim = trial.suggest_categorical('prod_emb_dim', [4, 8, 16, 32])
    n_layers     = trial.suggest_int('n_layers', 1, 3)
    hidden_dim   = trial.suggest_int('hidden_dim', 32, 256, step=32)
    emb_dropout  = trial.suggest_float('emb_dropout', 0.1, 0.6, step=0.1)
    mlp_dropout  = trial.suggest_float('mlp_dropout', 0.1, 0.5, step=0.1)
    lr           = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    batch_size   = trial.suggest_categorical('batch_size', [256, 512, 1024])
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-3, log=True)

    tr_loader, v_loader, _ = make_loaders(batch_size)

    model = FlexEmbeddingANN(
        n_users, n_products, len(FEATURE_COLS),
        user_emb_dim, prod_emb_dim, n_layers,
        hidden_dim, emb_dropout, mlp_dropout
    ).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    best_val  = float('inf')
    patience  = 0

    for epoch in range(N_EPOCHS_TRIAL):
        run_epoch(model, tr_loader, criterion, optimizer)
        val_rmse = run_epoch(model, v_loader, criterion)

        trial.report(val_rmse, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if val_rmse < best_val:
            best_val = val_rmse; patience = 0
        else:
            patience += 1
            if patience >= PATIENCE_TRIAL:
                break

    return best_val


In [ ]:
N_TRIALS = 50

study = optuna.create_study(
    direction = 'minimize',
    sampler   = TPESampler(seed=SEED),
    pruner    = MedianPruner(n_warmup_steps=5)
)

print(f"Starting Optuna — {N_TRIALS} trials...")
t0 = time.perf_counter()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
hpo_time = time.perf_counter() - t0

print(f"\nHPO done in {hpo_time/60:.1f} minutes")
print(f"Trials completed : {len(study.trials)}")
print(f"Trials pruned    : {sum(1 for t in study.trials if t.state == optuna.trial.TrialState.PRUNED)}")


In [ ]:
best = study.best_trial
print(f"Best val RMSE : {best.value:.4f}")
print("\nBest hyperparameters:")
for k, v in best.params.items():
    print(f"  {k:<20} = {v}")


In [ ]:
# Optuna hyperparameter importance
importances = optuna.importance.get_param_importances(study)
plt.figure(figsize=(8, 3))
plt.barh(list(importances.keys())[::-1], list(importances.values())[::-1], color='steelblue')
plt.xlabel('Relative importance')
plt.title('Optuna — Hyperparameter Importances (ANN)')
plt.tight_layout(); plt.show()


---
## Step 8 — Retrain Final Model on Best Params

In [ ]:
p = best.params
N_EPOCHS_FINAL = 30
PATIENCE_FINAL = 5

tr_loader, v_loader, te_loader = make_loaders(p['batch_size'])

final_model = FlexEmbeddingANN(
    n_users, n_products, len(FEATURE_COLS),
    p['user_emb_dim'], p['prod_emb_dim'],
    p['n_layers'], p['hidden_dim'],
    p['emb_dropout'], p['mlp_dropout']
).to(DEVICE)

optimizer = optim.Adam(final_model.parameters(),
                       lr=p['lr'], weight_decay=p['weight_decay'])
criterion = nn.MSELoss()

best_val       = float('inf')
best_epoch     = 0
patience_count = 0
best_state     = None
history        = {'train_rmse': [], 'val_rmse': []}

t0 = time.perf_counter()
for epoch in range(N_EPOCHS_FINAL):
    tr_rmse  = run_epoch(final_model, tr_loader, criterion, optimizer)
    val_rmse = run_epoch(final_model, v_loader,  criterion)
    history['train_rmse'].append(tr_rmse)
    history['val_rmse'].append(val_rmse)

    if val_rmse < best_val:
        best_val       = val_rmse
        best_epoch     = epoch + 1
        patience_count = 0
        best_state     = {k: v.clone() for k, v in final_model.state_dict().items()}
    else:
        patience_count += 1

    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:>2}/{N_EPOCHS_FINAL}  "
              f"Train: {tr_rmse}  Val: {val_rmse}")
    if patience_count >= PATIENCE_FINAL:
        print(f"Early stopping at epoch {epoch+1}")
        break

final_model.load_state_dict(best_state)
train_time = time.perf_counter() - t0
print(f"\nBest val RMSE: {best_val} at epoch {best_epoch}")
print(f"Training time: {train_time:.1f}s")


---
## Step 9 — Evaluate

In [ ]:
tr_rmse,  tr_mae  = run_epoch(final_model, tr_loader,  criterion), None
val_rmse, val_mae = run_epoch(final_model, v_loader,   criterion), None
te_rmse,  te_mae  = run_epoch(final_model, te_loader,  criterion), None

def full_metrics(model, loader, criterion):
    model.eval()
    preds_all, labels_all = [], []
    with torch.no_grad():
        for num_b, u_b, p_b, y_b in loader:
            num_b,u_b,p_b = num_b.to(DEVICE),u_b.to(DEVICE),p_b.to(DEVICE)
            preds_all.extend(model(num_b,u_b,p_b).cpu().numpy())
            labels_all.extend(y_b.numpy())
    p = np.array(preds_all); l = np.array(labels_all)
    return (round(float(np.sqrt(mean_squared_error(l,p))),4),
            round(float(mean_absolute_error(l,p)),4))

tr_rmse,  tr_mae  = full_metrics(final_model, tr_loader,  criterion)
val_rmse, val_mae = full_metrics(final_model, v_loader,   criterion)
te_rmse,  te_mae  = full_metrics(final_model, te_loader,  criterion)

print("=" * 55)
print("EmbeddingANN (Optimised) — Final Results")
print("=" * 55)
print(f"  Train      RMSE: {tr_rmse}   MAE: {tr_mae}")
print(f"  Validation RMSE: {val_rmse}   MAE: {val_mae}")
print(f"  Test       RMSE: {te_rmse}   MAE: {te_mae}")
print(f"  Train time : {train_time:.1f}s")
print(f"  Best epoch : {best_epoch}")
print(f"  Train-Test gap : {round(te_rmse-tr_rmse,4)}")


---
## Step 10 — Save Best Embeddings & Results

In [ ]:
# Save optimised embeddings
user_weights_opt = final_model.user_emb.weight.detach().cpu().numpy()
prod_weights_opt = final_model.prod_emb.weight.detach().cpu().numpy()

np.save(os.path.join(OUT_DIR, 'user_embeddings.npy'),    user_weights_opt)
np.save(os.path.join(OUT_DIR, 'product_embeddings.npy'), prod_weights_opt)

# Copy encoders
import shutil
for fname in ['user_encoder.json','product_encoder.json']:
    shutil.copy(os.path.join(EMB_DIR, fname), os.path.join(OUT_DIR, fname))

# Save results
result = {
    'model': 'EmbeddingANN (Optimised)',
    'train_rmse': tr_rmse, 'val_rmse': val_rmse, 'test_rmse': te_rmse,
    'train_mae':  tr_mae,  'val_mae':  val_mae,  'test_mae':  te_mae,
    'train_time_s': round(train_time, 1),
    'best_epoch': best_epoch,
    'best_params': best.params
}
with open(os.path.join(OUT_DIR, 'ann_results.json'), 'w') as f:
    json.dump(result, f, indent=2)

print(f"Saved to: {OUT_DIR}")
for fname in os.listdir(OUT_DIR):
    print(f"  {fname}")
